In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.utils import save_image

from PIL import Image
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

In [2]:
class ZeroDCEPP(nn.Module):

    def __init__(self):

        super(ZeroDCEPP,self).__init__()

        self.relu = nn.ReLU(inplace=True)

        number_f = 16

        self.conv1 = nn.Conv2d(3,number_f,3,1,1)
        self.conv2 = nn.Conv2d(number_f,number_f,3,1,1)
        self.conv3 = nn.Conv2d(number_f,number_f,3,1,1)
        self.conv4 = nn.Conv2d(number_f,number_f,3,1,1)

        self.conv5 = nn.Conv2d(number_f*2,number_f,3,1,1)
        self.conv6 = nn.Conv2d(number_f*2,number_f,3,1,1)

        self.conv7 = nn.Conv2d(number_f*2,24,3,1,1)

    def forward(self,x):

        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))
        x4 = self.relu(self.conv4(x3))

        x5 = self.relu(self.conv5(torch.cat([x3,x4],1)))
        x6 = self.relu(self.conv6(torch.cat([x2,x5],1)))

        x_r = torch.tanh(self.conv7(torch.cat([x1,x6],1)))

        r1,r2,r3,r4,r5,r6,r7,r8 = torch.split(x_r,3,dim=1)

        x = x + r1*(x*x - x)
        x = x + r2*(x*x - x)
        x = x + r3*(x*x - x)
        x = x + r4*(x*x - x)
        x = x + r5*(x*x - x)
        x = x + r6*(x*x - x)
        x = x + r7*(x*x - x)

        enhanced = x + r8*(x*x - x)

        enhanced = torch.clamp(enhanced,0,1)

        return enhanced,x_r

In [ ]:
class L_spa(nn.Module):

    def __init__(self):

        super(L_spa,self).__init__()

        kernel_left = torch.FloatTensor([[0,0,0],[-1,1,0],[0,0,0]]).unsqueeze(0).unsqueeze(0)
        kernel_right = torch.FloatTensor([[0,0,0],[0,1,-1],[0,0,0]]).unsqueeze(0).unsqueeze(0)
        kernel_up = torch.FloatTensor([[0,-1,0],[0,1,0],[0,0,0]]).unsqueeze(0).unsqueeze(0)
        kernel_down = torch.FloatTensor([[0,0,0],[0,1,0],[0,-1,0]]).unsqueeze(0).unsqueeze(0)

        self.weight_left = nn.Parameter(kernel_left,requires_grad=False)
        self.weight_right = nn.Parameter(kernel_right,requires_grad=False)
        self.weight_up = nn.Parameter(kernel_up,requires_grad=False)
        self.weight_down = nn.Parameter(kernel_down,requires_grad=False)

    def forward(self,org,enhance):

        org_mean = torch.mean(org,1,keepdim=True)
        enhance_mean = torch.mean(enhance,1,keepdim=True)

        D_org_left = F.conv2d(org_mean,self.weight_left,padding=1)
        D_org_right = F.conv2d(org_mean,self.weight_right,padding=1)
        D_org_up = F.conv2d(org_mean,self.weight_up,padding=1)
        D_org_down = F.conv2d(org_mean,self.weight_down,padding=1)

        D_enh_left = F.conv2d(enhance_mean,self.weight_left,padding=1)
        D_enh_right = F.conv2d(enhance_mean,self.weight_right,padding=1)
        D_enh_up = F.conv2d(enhance_mean,self.weight_up,padding=1)
        D_enh_down = F.conv2d(enhance_mean,self.weight_down,padding=1)

        loss = torch.mean((D_org_left-D_enh_left)**2 +
                          (D_org_right-D_enh_right)**2 +
                          (D_org_up-D_enh_up)**2 +
                          (D_org_down-D_enh_down)**2)

        return loss

In [ ]:
class L_exp(nn.Module):

    def __init__(self,patch_size=16,mean_val=0.6):

        super(L_exp,self).__init__()

        self.pool = nn.AvgPool2d(patch_size)
        self.mean_val = mean_val

    def forward(self,x):

        b = torch.mean(x,1,keepdim=True)

        mean = self.pool(b)

        loss = torch.mean((mean-self.mean_val)**2)

        return loss

In [ ]:
class L_color(nn.Module):

    def __init__(self):

        super(L_color,self).__init__()

    def forward(self,x):

        mean_rgb = torch.mean(x,[2,3],keepdim=True)

        mr,mg,mb = torch.split(mean_rgb,1,dim=1)

        Drg = (mr-mg)**2
        Drb = (mr-mb)**2
        Dgb = (mb-mg)**2

        loss = torch.sqrt(Drg**2 + Drb**2 + Dgb**2)

        return torch.mean(loss)

In [ ]:
class L_TV(nn.Module):

    def __init__(self):

        super(L_TV,self).__init__()

    def forward(self,x):

        batch_size = x.size()[0]

        h_x = x.size()[2]
        w_x = x.size()[3]

        count_h = (x.size()[2]-1)*x.size()[3]
        count_w = x.size()[2]*(x.size()[3]-1)

        h_tv = torch.pow((x[:,:,1:,:]-x[:,:,:h_x-1,:]),2).sum()
        w_tv = torch.pow((x[:,:,:,1:]-x[:,:,:,:w_x-1]),2).sum()

        return (h_tv/count_h + w_tv/count_w)/batch_size

In [ ]:
class LowLightDataset(Dataset):

    def __init__(self,image_path):

        self.images = glob.glob(os.path.join(image_path,"*.png")) + \
                      glob.glob(os.path.join(image_path,"*.jpg"))

        self.transform = transforms.Compose([
            transforms.Resize((256,256)),
            transforms.ToTensor()
        ])

        print("Images found:",len(self.images))

    def __len__(self):

        return len(self.images)

    def __getitem__(self,index):

        img = Image.open(self.images[index]).convert("RGB")

        img = self.transform(img)

        return img

In [ ]:
def train(data_path):

    import matplotlib.pyplot as plt
    import random

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = ZeroDCEPP().to(device)

    dataset = LowLightDataset(data_path)

    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    L_color_loss = L_color().to(device)
    L_exp_loss = L_exp().to(device)
    L_tv_loss = L_TV().to(device)
    L_spa_loss = L_spa().to(device)

    epochs = 1000

    # 🔹 choose 10 random samples for monitoring
    sample_indices = random.sample(range(len(dataset)), 10)
    sample_imgs = torch.stack([dataset[i] for i in sample_indices]).to(device)

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for img in dataloader:

            img = img.to(device)

            enhanced, x_r = model(img)

            loss_spa = L_spa_loss(img, enhanced)
            loss_exp = 10 * L_exp_loss(enhanced)
            loss_col = 5 * L_color_loss(enhanced)
            loss_tv = 200 * L_tv_loss(x_r)

            loss = loss_spa + loss_exp + loss_col + loss_tv

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)

        print("Epoch:", epoch, "Loss:", avg_loss)

        # 🔹 show 10 samples every 20 epochs
        if epoch % 20 == 0:

            model.eval()

            with torch.no_grad():

                enhanced_batch, _ = model(sample_imgs)

                plt.figure(figsize=(10,20))

                for i in range(10):

                    inp = sample_imgs[i].cpu().permute(1,2,0).numpy()
                    out = enhanced_batch[i].cpu().permute(1,2,0).numpy()

                    # input
                    plt.subplot(10,2,2*i+1)
                    plt.imshow(inp)
                    plt.title("Input")
                    plt.axis("off")

                    # enhanced
                    plt.subplot(10,2,2*i+2)
                    plt.imshow(out)
                    plt.title(f"Epoch {epoch}")
                    plt.axis("off")

                plt.tight_layout()
                plt.show()

        if epoch % 50 == 0:
            torch.save(model.state_dict(), f"zerodce_epoch_{epoch}.pth")

    torch.save(model.state_dict(), "zerodce_final.pth")

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("soumikrakshit/lol-dataset")

data_path = os.path.join(path,"lol_dataset","our485","low")

train(data_path)

In [ ]:
model = ZeroDCEPP()

model.load_state_dict(torch.load("zerodce_final.pth"))

model.eval()

img = Image.open("test.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

img = transform(img).unsqueeze(0)

with torch.no_grad():

    enhanced,_ = model(img)

save_image(enhanced,"enhanced.png")